# Phishing Websites: Data Preparation

## Overview

1. Load and verify the raw data (no cleaning needed - dataset has no missing values).
2. Create a single, hold-out test set.
3. From the main training set, generate multiple training datasets with varying **imbalance ratios (IR)** by undersampling the minority class.
4. For each IR, create **multiple repetitions** with different random samples of the minority class.
5. For each IR and repetition, create a size-matched **control dataset** with the original class ratio.
6. Preprocess and save all generated datasets.

## Dataset Information

- **OpenML ID:** 4534
- **Instances:** 11,055 websites
- **Features:** 30 (all categorical with values -1, 0, or 1)
- **Target:** Result (-1 = Phishing, 1 = Legitimate)
- **Natural Imbalance Ratio:** ~1.26:1 (Legitimate sites are majority)
- **Missing Values:** 0 (clean dataset)

# 1. Dataset Configuration

In [1]:
DATASET_NAME = "phishing"

print(f"Dataset: {DATASET_NAME}")

Dataset: phishing


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from pathlib import Path
import sys

sys.path.append(str(Path("../../").resolve()))
from config.config import get_config

config = get_config()

RAW_PATH = Path("../../data/raw/phising.csv")
PROCESSED_PATH = Path(f"../../data/processed/{DATASET_NAME}/")

TARGET_FEATURE = "target"
ORIGINAL_TARGET = "Result"
CLASS_PHISHING = 0    # Minority class (will be mapped from -1)
CLASS_LEGITIMATE = 1  # Majority class (stays as 1)

RANDOM_STATE = config.experiment.random_state
IMBALANCE_RATIOS = config.experiment.imbalance_ratios
N_REPETITIONS = config.experiment.n_repetitions

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  - Dataset: {DATASET_NAME}")
print(f"  - Raw data path: {RAW_PATH}")
print(f"  - Processed data path: {PROCESSED_PATH}")
print(f"  - Target feature: {TARGET_FEATURE}")
print(f"  - Imbalance ratios: {IMBALANCE_RATIOS}")
print(f"  - Repetitions per IR: {N_REPETITIONS}")
print(f"  - Random state: {RANDOM_STATE}")
print(f"\nLoading the raw dataset...")

df_raw = pd.read_csv(RAW_PATH)

print(f"Raw dataset loaded. Shape: {df_raw.shape}")

Configuration:
  - Dataset: phishing
  - Raw data path: ../../data/raw/phising.csv
  - Processed data path: ../../data/processed/phishing
  - Target feature: target
  - Imbalance ratios: [1, 3, 5, 7, 10, 20, 50, 100]
  - Repetitions per IR: 10
  - Random state: 42

Loading the raw dataset...
Raw dataset loaded. Shape: (11055, 32)


In [3]:
# Drop the id column as it's not a feature
if 'id' in df_raw.columns:
    df_raw = df_raw.drop('id', axis=1)
    print("Dropped 'id' column")

print(f"\nDataset shape after dropping id: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")

Dropped 'id' column

Dataset shape after dropping id: (11055, 31)

Columns: ['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report', 'Result']


# 2. Data Verification & Target Encoding

The Phishing Websites dataset has no missing values. We need to:
1. Verify data integrity
2. Re-encode the target variable from {-1, 1} to {0, 1} for consistency with other datasets

In [4]:
# Verify no missing values
print("Missing values per column:")
missing = df_raw.isnull().sum()
print(f"Total missing values: {missing.sum()}")

Missing values per column:
Total missing values: 0


In [5]:
# Check unique values in the target
print(f"\nOriginal target ('{ORIGINAL_TARGET}') unique values: {sorted(df_raw[ORIGINAL_TARGET].unique())}")
print(f"\nOriginal target distribution:")
print(df_raw[ORIGINAL_TARGET].value_counts().sort_index())


Original target ('Result') unique values: [-1, 1]

Original target distribution:
Result
-1    4898
 1    6157
Name: count, dtype: int64


In [6]:
# Re-encode target: -1 (Phishing) -> 0, 1 (Legitimate) -> 1
df_cleaned = df_raw.copy()

# Map the target variable
df_cleaned[TARGET_FEATURE] = df_cleaned[ORIGINAL_TARGET].map({-1: 0, 1: 1})

# Drop the original target column
df_cleaned = df_cleaned.drop(ORIGINAL_TARGET, axis=1)

print(f"Target variable re-encoded:")
print(f"  -1 (Phishing)   -> 0")
print(f"   1 (Legitimate) -> 1")

print(f"\nNew target distribution:")
print(df_cleaned[TARGET_FEATURE].value_counts().sort_index())

Target variable re-encoded:
  -1 (Phishing)   -> 0
   1 (Legitimate) -> 1

New target distribution:
target
0    4898
1    6157
Name: count, dtype: int64


# 3. Confirming Majority and Minority Classes

For our imbalance experiments:
- **Class 0 (Minority):** Phishing websites (4,898 instances, ~44.3%)
- **Class 1 (Majority):** Legitimate websites (6,157 instances, ~55.7%)
- **Natural Imbalance Ratio:** ~1.26:1 (Legitimate sites are majority)

In [7]:
print("Target variable distribution:")
print(df_cleaned[TARGET_FEATURE].value_counts().sort_index())

n_phishing = df_cleaned[TARGET_FEATURE].value_counts()[CLASS_PHISHING]
n_legitimate = df_cleaned[TARGET_FEATURE].value_counts()[CLASS_LEGITIMATE]

print(f"\nClass balance:")
print(f"  - Phishing (0):   {n_phishing:,} ({n_phishing/len(df_cleaned)*100:.2f}%)")
print(f"  - Legitimate (1): {n_legitimate:,} ({n_legitimate/len(df_cleaned)*100:.2f}%)")

print(f"\nMajority class: Legitimate (1)")
print(f"Minority class: Phishing (0)")
CLASS_MAJORITY = CLASS_LEGITIMATE
CLASS_MINORITY = CLASS_PHISHING

natural_ir = max(n_phishing, n_legitimate) / min(n_phishing, n_legitimate)
print(f"\nNatural imbalance ratio: {natural_ir:.2f}:1")

Target variable distribution:
target
0    4898
1    6157
Name: count, dtype: int64

Class balance:
  - Phishing (0):   4,898 (44.31%)
  - Legitimate (1): 6,157 (55.69%)

Majority class: Legitimate (1)
Minority class: Phishing (0)

Natural imbalance ratio: 1.26:1


# 4. Create a Hold-Out Test Set

We perform a one-time stratified split to create a final test set. All experimental datasets will be generated from the `train_full_df`.

In [8]:
X = df_cleaned.drop(TARGET_FEATURE, axis=1)
y = df_cleaned[[TARGET_FEATURE]]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=config.experiment.test_size,
    random_state=RANDOM_STATE,
    stratify=y
)

train_full_df = pd.concat([X_train_full, y_train_full], axis=1)

print(f"Full training set shape: {train_full_df.shape}")
print(f"Hold-out test set shape: {X_test.shape}")
print(f"\nTraining set class distribution:")
print(train_full_df[TARGET_FEATURE].value_counts().sort_index())

n_train_majority = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MAJORITY].shape[0]
n_train_minority = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MINORITY].shape[0]
print(f"\nTraining set: {n_train_majority:,} majority, {n_train_minority:,} minority")

Full training set shape: (8844, 31)
Hold-out test set shape: (2211, 30)

Training set class distribution:
target
0    3918
1    4926
Name: count, dtype: int64

Training set: 4,926 majority, 3,918 minority


# 5. Generate Imbalanced and Control Datasets with Multiple Repetitions

For each IR, we create multiple repetitions by sampling different subsets of the minority class. This allows us to test whether methods work reliably across different minority class samples.

**Methodology:**
1. Start with the **full majority class** (Legitimate websites).
2. **Undersample the minority class** (Phishing websites) to achieve the desired Imbalance Ratio (IR).
3. **Repeat this sampling N_REPETITIONS times** with different random seeds.
4. Create a size-matched **control dataset** for each IR and repetition.

In [9]:
minority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MINORITY]  # Phishing
majority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MAJORITY]  # Legitimate
n_minority_available = len(minority_df)
n_majority_available = len(majority_df)

print(f"\nFull training set composition:")
print(f"  - Majority (Legitimate): {n_majority_available:,}")
print(f"  - Minority (Phishing):   {n_minority_available:,}")
print(f"\nGenerating datasets with {N_REPETITIONS} repetitions per imbalance ratio...")

generated_datasets = {}
skipped_ratios = []

for ir in IMBALANCE_RATIOS:
    print(f"\nProcessing Imbalance Ratio (IR) = {ir}:1")
    
    for rep_id in range(1, N_REPETITIONS + 1):
        # Use different random seed for each repetition
        rep_seed = RANDOM_STATE + (ir * 1000) + rep_id
        
        if ir == 1:
            # For IR=1:1, undersample majority to match minority
            majority_undersampled = resample(
                majority_df,
                replace=False,
                n_samples=n_minority_available, 
                random_state=rep_seed 
            )
            imbalanced_df = pd.concat([majority_undersampled, minority_df])
            
        else:
            # For IR > 1, keep full majority and undersample minority
            majority_full_set = majority_df
            
            n_minority_imbalanced = int(n_majority_available / ir)

            if n_minority_imbalanced > n_minority_available:
                if rep_id == 1:  # Only print once
                    print(f"  SKIPPING IR={ir}: Requires {n_minority_imbalanced} minority samples but only {n_minority_available} available.")
                    skipped_ratios.append(ir)
                continue
            if n_minority_imbalanced < 1:
                if rep_id == 1:
                    print(f"  SKIPPING IR={ir}: Would result in zero minority samples.")
                    skipped_ratios.append(ir)
                continue

            minority_undersampled = resample(
                minority_df,
                replace=False,
                n_samples=n_minority_imbalanced,
                random_state=rep_seed 
            )

            imbalanced_df = pd.concat([majority_full_set, minority_undersampled])

        total_size = len(imbalanced_df)
        
        dataset_key = f'imbalanced_ir_{ir}_rep{rep_id}'
        generated_datasets[dataset_key] = imbalanced_df
        
        n_maj = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_MAJORITY])
        n_min = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_MINORITY])
        
        if rep_id == 1:  # Only print first repetition details
            print(f"  Rep {rep_id}: {total_size:,} samples ({n_maj:,} majority, {n_min:,} minority)")

        # Create control dataset with original class ratio
        if total_size >= len(train_full_df):
            control_df = train_full_df.copy()
        else:
            control_df, _ = train_test_split(
                train_full_df,
                train_size=total_size,
                random_state=rep_seed,  
                stratify=train_full_df[TARGET_FEATURE]
            )
        
        control_key = f'control_ir_{ir}_rep{rep_id}'
        generated_datasets[control_key] = control_df

print(f"Total datasets created: {len(generated_datasets)}")
print(f"  - Imbalanced: {len([k for k in generated_datasets.keys() if 'imbalanced' in k])}")
print(f"  - Control: {len([k for k in generated_datasets.keys() if 'control' in k])}")

if skipped_ratios:
    print(f"\nSkipped imbalance ratios (insufficient minority samples): {list(set(skipped_ratios))}")


Full training set composition:
  - Majority (Legitimate): 4,926
  - Minority (Phishing):   3,918

Generating datasets with 10 repetitions per imbalance ratio...

Processing Imbalance Ratio (IR) = 1:1
  Rep 1: 7,836 samples (3,918 majority, 3,918 minority)

Processing Imbalance Ratio (IR) = 3:1
  Rep 1: 6,568 samples (4,926 majority, 1,642 minority)

Processing Imbalance Ratio (IR) = 5:1
  Rep 1: 5,911 samples (4,926 majority, 985 minority)

Processing Imbalance Ratio (IR) = 7:1
  Rep 1: 5,629 samples (4,926 majority, 703 minority)

Processing Imbalance Ratio (IR) = 10:1
  Rep 1: 5,418 samples (4,926 majority, 492 minority)

Processing Imbalance Ratio (IR) = 20:1
  Rep 1: 5,172 samples (4,926 majority, 246 minority)

Processing Imbalance Ratio (IR) = 50:1
  Rep 1: 5,024 samples (4,926 majority, 98 minority)

Processing Imbalance Ratio (IR) = 100:1
  Rep 1: 4,975 samples (4,926 majority, 49 minority)
Total datasets created: 160
  - Imbalanced: 80
  - Control: 80


In [10]:


summary_data = []
for name in sorted(generated_datasets.keys()):
    df_temp = generated_datasets[name]
    n_total = len(df_temp)
    n_maj = len(df_temp[df_temp[TARGET_FEATURE] == CLASS_MAJORITY])
    n_min = len(df_temp[df_temp[TARGET_FEATURE] == CLASS_MINORITY])
    actual_ir = n_maj / n_min if n_min > 0 else float('inf')
    
    summary_data.append({
        'Dataset': name,
        'Total': n_total,
        'Majority': n_maj,
        'Minority': n_min,
        'Actual IR': f"{actual_ir:.2f}:1"
    })

summary_df = pd.DataFrame(summary_data)

# Show first 10 datasets as sample
print("\nFirst 10 datasets:")
display(summary_df.head(10))

print(f"\n... and {len(summary_df) - 10} more datasets")


First 10 datasets:


,Dataset,Total,Majority,Minority,Actual IR
0,control_ir_100_rep1,4975,2771,2204,1.26:1
1,control_ir_100_rep10,4975,2771,2204,1.26:1
2,control_ir_100_rep2,4975,2771,2204,1.26:1
3,control_ir_100_rep3,4975,2771,2204,1.26:1
4,control_ir_100_rep4,4975,2771,2204,1.26:1
5,control_ir_100_rep5,4975,2771,2204,1.26:1
6,control_ir_100_rep6,4975,2771,2204,1.26:1
7,control_ir_100_rep7,4975,2771,2204,1.26:1
8,control_ir_100_rep8,4975,2771,2204,1.26:1
9,control_ir_100_rep9,4975,2771,2204,1.26:1



... and 150 more datasets


# 6. Preprocessing and Saving All Datasets

All features are categorical with values {-1, 0, 1}. We apply StandardScaler for consistency with other datasets in the experiment pipeline.

We fit the scaler **once** on the full training data. Then, we transform all generated training sets and the hold-out test set using this single, consistent scaler.

In [11]:
FEATURES_TO_SCALE = [col for col in df_cleaned.columns if col != TARGET_FEATURE]

print(f"Features to scale ({len(FEATURES_TO_SCALE)}):")
for i, feat in enumerate(FEATURES_TO_SCALE, 1):
    print(f"  {i:2d}. {feat}")

Features to scale (30):
   1. having_IP_Address
   2. URL_Length
   3. Shortining_Service
   4. having_At_Symbol
   5. double_slash_redirecting
   6. Prefix_Suffix
   7. having_Sub_Domain
   8. SSLfinal_State
   9. Domain_registeration_length
  10. Favicon
  11. port
  12. HTTPS_token
  13. Request_URL
  14. URL_of_Anchor
  15. Links_in_tags
  16. SFH
  17. Submitting_to_email
  18. Abnormal_URL
  19. Redirect
  20. on_mouseover
  21. RightClick
  22. popUpWidnow
  23. Iframe
  24. age_of_domain
  25. DNSRecord
  26. web_traffic
  27. Page_Rank
  28. Google_Index
  29. Links_pointing_to_page
  30. Statistical_report


In [12]:
scaler = StandardScaler()
scaler.fit(X_train_full[FEATURES_TO_SCALE])

print("Scaler fitted on full training data.")
print(f"\nScaler statistics (first 5 features):")
for i, feat in enumerate(FEATURES_TO_SCALE[:5]):
    print(f"  {feat}: mean={scaler.mean_[i]:.4f}, std={scaler.scale_[i]:.4f}")

Scaler fitted on full training data.

Scaler statistics (first 5 features):
  having_IP_Address: mean=0.3180, std=0.9481
  URL_Length: mean=-0.6340, std=0.7664
  Shortining_Service: mean=0.7415, std=0.6709
  having_At_Symbol: mean=0.7004, std=0.7138
  double_slash_redirecting: mean=0.7431, std=0.6692


In [13]:
print("Scaling and saving datasets...\n")

saved_count = 0
for name, df in generated_datasets.items():
    X_temp = df.drop(columns=[TARGET_FEATURE])
    y_temp = df[[TARGET_FEATURE]]

    X_processed = scaler.transform(X_temp[FEATURES_TO_SCALE])
    X_processed_df = pd.DataFrame(X_processed, columns=FEATURES_TO_SCALE)
    
    final_df = X_processed_df.reset_index(drop=True)
    final_df[TARGET_FEATURE] = y_temp.reset_index(drop=True)
    
    save_path = PROCESSED_PATH / f"train_{name}.csv"
    final_df.to_csv(save_path, index=False)
    saved_count += 1
    
    # Print progress every 20 files
    if saved_count % 20 == 0:
        print(f"  Saved {saved_count} files...")

print(f"\nSaved all {saved_count} training files.")

Scaling and saving datasets...

  Saved 20 files...
  Saved 40 files...
  Saved 60 files...
  Saved 80 files...
  Saved 100 files...
  Saved 120 files...
  Saved 140 files...
  Saved 160 files...

Saved all 160 training files.


In [14]:
# Process and save test set
X_test_processed = scaler.transform(X_test[FEATURES_TO_SCALE])
X_test_processed_df = pd.DataFrame(X_test_processed, columns=FEATURES_TO_SCALE)

test_df = X_test_processed_df.reset_index(drop=True)
test_df[TARGET_FEATURE] = y_test.reset_index(drop=True)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print(f"Saved test set: test.csv ({len(test_df):,} samples)")
print(f"\nTest set class distribution:")
print(test_df[TARGET_FEATURE].value_counts().sort_index())

Saved test set: test.csv (2,211 samples)

Test set class distribution:
target
0     980
1    1231
Name: count, dtype: int64


# 7. Save Metadata

In [15]:
import json
from datetime import datetime

# Determine which IRs were actually created
created_irs = sorted(list(set([int(k.split('_')[2]) for k in generated_datasets.keys() if 'imbalanced' in k])))

# Save metadata about this dataset processing
metadata = {
    "dataset_name": DATASET_NAME,
    "openml_id": 4534,
    "target_feature": TARGET_FEATURE,
    "original_target": ORIGINAL_TARGET,
    "processing_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_train_files": len(generated_datasets),
    "imbalance_ratios_requested": IMBALANCE_RATIOS,
    "imbalance_ratios_created": created_irs,
    "n_repetitions": N_REPETITIONS,
    "random_state": RANDOM_STATE,
    "test_size": len(test_df),
    "train_full_size": len(train_full_df),
    "n_features": len(FEATURES_TO_SCALE),
    "features": FEATURES_TO_SCALE,
    "class_mapping": {
        "0": "Phishing (minority)",
        "1": "Legitimate (majority)"
    },
    "original_class_encoding": {
        "-1": "Phishing -> mapped to 0",
        "1": "Legitimate -> stays as 1"
    },
    "natural_imbalance_ratio": round(natural_ir, 2),
    "notes": "Phishing Websites detection dataset - 30 categorical features extracted from URLs and website content"
}

metadata_path = PROCESSED_PATH / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
print("\nMetadata contents:")
for key, value in metadata.items():
    if key != 'features':  # Don't print all features
        print(f"  {key}: {value}")

Metadata saved to: ../../data/processed/phishing/metadata.json

Metadata contents:
  dataset_name: phishing
  openml_id: 4534
  target_feature: target
  original_target: Result
  processing_timestamp: 2026-01-07 15:42:43
  n_train_files: 160
  imbalance_ratios_requested: [1, 3, 5, 7, 10, 20, 50, 100]
  imbalance_ratios_created: [1, 3, 5, 7, 10, 20, 50, 100]
  n_repetitions: 10
  random_state: 42
  test_size: 2211
  train_full_size: 8844
  n_features: 30
  class_mapping: {'0': 'Phishing (minority)', '1': 'Legitimate (majority)'}
  original_class_encoding: {'-1': 'Phishing -> mapped to 0', '1': 'Legitimate -> stays as 1'}
  natural_imbalance_ratio: 1.26
  notes: Phishing Websites detection dataset - 30 categorical features extracted from URLs and website content


# 8. Verification & Summary

In [16]:
# Verify saved files
saved_files = list(PROCESSED_PATH.glob("*.csv"))
print(f"\nVerification:")
print(f"  - Total CSV files saved: {len(saved_files)}")
print(f"  - Expected: {len(generated_datasets) + 1} (training + test)")

# Check file sizes
total_size_mb = sum(f.stat().st_size for f in saved_files) / (1024 * 1024)
print(f"  - Total size: {total_size_mb:.2f} MB")


Verification:
  - Total CSV files saved: 161
  - Expected: 161 (training + test)
  - Total size: 518.68 MB


In [17]:
# Load and verify a sample file
sample_file = PROCESSED_PATH / "train_imbalanced_ir_1_rep1.csv"
if sample_file.exists():
    sample_df = pd.read_csv(sample_file)
    print(f"\nSample verification (train_imbalanced_ir_1_rep1.csv):")
    print(f"  Shape: {sample_df.shape}")
    print(f"  Columns: {list(sample_df.columns)[:5]}... (and {len(sample_df.columns)-5} more)")
    print(f"  Target distribution:")
    print(sample_df[TARGET_FEATURE].value_counts().sort_index())
    print(f"\n  Feature statistics (first 3):")
    print(sample_df[FEATURES_TO_SCALE[:3]].describe().round(3))


Sample verification (train_imbalanced_ir_1_rep1.csv):
  Shape: (7836, 31)
  Columns: ['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting']... (and 26 more)
  Target distribution:
target
0    3918
1    3918
Name: count, dtype: int64

  Feature statistics (first 3):
       having_IP_Address  URL_Length  Shortining_Service
count           7836.000    7836.000            7836.000
mean              -0.010      -0.006               0.008
std                1.004       0.995               0.991
min               -1.390      -0.478              -2.596
25%               -1.390      -0.478               0.385
50%                0.719      -0.478               0.385
75%                0.719      -0.478               0.385
max                0.719       2.132               0.385


In [18]:


print(f"\nDataset: {DATASET_NAME}")
print(f"\nOriginal data:")
print(f"  - Total instances: {len(df_cleaned):,}")
print(f"  - Features: {len(FEATURES_TO_SCALE)}")
print(f"  - Natural IR: {natural_ir:.2f}:1")

print(f"\nGenerated datasets:")
print(f"  - Training files: {len(generated_datasets)}")
print(f"  - Test file: 1")
print(f"  - Imbalance ratios: {created_irs}")
print(f"  - Repetitions per IR: {N_REPETITIONS}")

print(f"\nOutput location: {PROCESSED_PATH}")
print(f"\nProcessing complete! All datasets are ready for experiments.")


Dataset: phishing

Original data:
  - Total instances: 11,055
  - Features: 30
  - Natural IR: 1.26:1

Generated datasets:
  - Training files: 160
  - Test file: 1
  - Imbalance ratios: [1, 3, 5, 7, 10, 20, 50, 100]
  - Repetitions per IR: 10

Output location: ../../data/processed/phishing

Processing complete! All datasets are ready for experiments.
